In [1]:
import pandas as pd

In [2]:
import os
print(os.getcwd())

C:\Users\I335750\Downloads\Capstone project\RAG_Project


In [ ]:


file_path = r"C:\Users\I335750\Downloads\Capstone project\RAG_Project\embedding_ready_reviews.parquet"

df = pd.read_parquet(file_path)

print(df.shape)
df.head()

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 758479 entries, 0 to 758478
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   parent_asin          758479 non-null  object 
 1   asin                 758479 non-null  object 
 2   text                 758479 non-null  object 
 3   rating               758479 non-null  float64
 4   helpful_vote         758479 non-null  int64  
 5   verified_purchase    758479 non-null  bool   
 6   timestamp            758479 non-null  int64  
 7   review_title         758479 non-null  object 
 8   product_title        758479 non-null  object 
 9   description          758479 non-null  object 
 10  features             758479 non-null  object 
 11  categories           758479 non-null  object 
 12  average_rating       758479 non-null  float64
 13  rating_number        758479 non-null  int64  
 14  price                572675 non-null  float64
 15  store            

In [8]:
#Creating a smaller parquet file (10k–50k rows).
#Create a smaller subset
#Save the smaller parquet file
#Load the small dataset for your RAG pipeline

In [4]:
df_small = df.sample(n=20000, random_state=42)

In [5]:
df_small.to_parquet("embedding_ready_reviews_small.parquet", index=False)

In [2]:
df = pd.read_parquet("embedding_ready_reviews_small.parquet")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   parent_asin          20000 non-null  object 
 1   asin                 20000 non-null  object 
 2   text                 20000 non-null  object 
 3   rating               20000 non-null  float64
 4   helpful_vote         20000 non-null  int64  
 5   verified_purchase    20000 non-null  bool   
 6   timestamp            20000 non-null  int64  
 7   review_title         20000 non-null  object 
 8   product_title        20000 non-null  object 
 9   description          20000 non-null  object 
 10  features             20000 non-null  object 
 11  categories           20000 non-null  object 
 12  average_rating       20000 non-null  float64
 13  rating_number        20000 non-null  int64  
 14  price                15097 non-null  float64
 15  store                19937 non-null 

### sentiment analysis

In [4]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def get_sentiment(text):
    try:
        result = sentiment_model(text[:512])[0]
        return result['label'], result['score']
    except:
        return "neutral", 0.0

df[['sentiment','sentiment_score']] = df['clean_review_text'].apply(
    lambda x: pd.Series(get_sentiment(str(x)))
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [5]:
df.head(5)

,parent_asin,asin,text,rating,helpful_vote,verified_purchase,timestamp,review_title,product_title,description,...,store,details,clean_review_text,clean_product_title,clean_features,clean_categories,clean_description,embedding_text,sentiment,sentiment_score
0,B09XMP1M76,B07PHJK6GY,Helps to keep s parts of the stove clean,5.0,0,True,1672689232853,Great product,MOACOCK 50 Pack Aluminum Foil Square Stove Bur...,[Put the Foil Gas Burner Liners on top of the ...,...,MOACOCK,"{'': None, 'AC Adapter Current': None, 'Access...",helps to keep s parts of the stove clean,moacock 50 pack aluminum foil square stove bur...,,,,moacock 50 pack aluminum foil square stove bur...,POSITIVE,0.988581
1,B0BV15TJHN,B09PQHTL27,The hoses look like they are made well; very e...,5.0,0,False,1658700798198,Nice quality stainless braided hoses,Premium Stainless Steel Washing Machine Hoses ...,[],...,WELLUP,"{'': None, 'AC Adapter Current': None, 'Access...",the hoses look like they are made well very ea...,premium stainless steel washing machine hoses ...,,,,premium stainless steel washing machine hoses ...,POSITIVE,0.996874
2,B0C2Z4QHXG,B07P9NP399,This water filter replacement works great! La...,5.0,0,True,1611494366184,Perfect filter replacement for fridge,GLACIER FRESH XWF Replacement for GE XWF Refri...,[],...,GLACIER FRESH,"{'': None, 'AC Adapter Current': None, 'Access...",this water filter replacement works great last...,glacier fresh xwf replacement for ge xwf refri...,,,,glacier fresh xwf replacement for ge xwf refri...,NEGATIVE,0.989322
3,B0887RSP7S,B07T8XSDTB,Great Filter!!! Bought these to reduce the con...,5.0,0,True,1577914521678,Great filter,Stainless Steel Reusable Basket 8-12 Cup Sturd...,[],...,YEOSEN,"{'': None, 'AC Adapter Current': None, 'Access...",great filter bought these to reduce the consta...,stainless steel reusable basket 8-12 cup sturd...,,,,stainless steel reusable basket 8-12 cup sturd...,NEGATIVE,0.995535
4,B0121CFQ4E,B0121CFQ4E,Popped right in and appears to be working perf...,5.0,0,True,1473352455000,Easy replacement filter for your fridge.,Fette Filter - Refrigerator Compatible Air Fil...,[Refrigerator Compatible Air Filter Compatible...,...,Fette Filter,"{'': None, 'AC Adapter Current': None, 'Access...",popped right in and appears to be working perf...,fette filter - refrigerator compatible air fil...,,,,fette filter - refrigerator compatible air fil...,POSITIVE,0.999853


Load Embedding model

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

C:\Users\I335750\AppData\Local\Temp\ipykernel_29176\1397533967.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Apply Chunking to Your Dataset

In [21]:
#Create the text splitter

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
#from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # characters per chunk
    chunk_overlap=100    # overlap to maintain context
)

In [ ]:
## Convert your dataframe text into documents
#dataframe already has a column embedding_text, which is perfect for RAG.

In [11]:
from langchain_core.documents import Document

documents = []

for i, row in df.iterrows():
    documents.append(
        Document(
            page_content=row["embedding_text"],
            metadata={"asin": row["asin"]}
        )
    )

len(documents)

20000

In [ ]:
#LangChain Document object is created above, to understand what we have done above below is a short example.
#    Document(
#    page_content="This water filter works great...",
#    metadata={"asin": "B00123"}
#)

In [ ]:
#Split documents into chunks

In [12]:
chunks = text_splitter.split_documents(documents)

len(chunks)

23728

In [27]:
#Create vector embeddings and FAISS index
#convert chunks into vector embeddings.
#Text chunk → Embedding vector → FAISS index

In [13]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [44]:

vectorstore.save_local("faiss_index")

In [28]:
#Create a retriever
#Retriever is what RAG uses to search relevant chunks.
#For every query → return top 5 relevant chunks

In [14]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [ ]:
#Test retrieval

In [15]:
results = retriever.invoke("best water filter under 20000")

for r in results:
    print(r.page_content[:200])

aquacrest   1 4 the price of the smasung filters at home depot or lowes, crazy great tasting water.
frigidaire eptwfu01 water filtration filter, 1 count, white   price is good
this brand again and can recommend this to anyone looking for a water filter. i received a discount for my honest review.
water filter   great product ..worked perfect
frigidaire eptwfu01 water filtration filter, 1 count, white   works well and easy to install but a little pricey


In [35]:
#Create Prompt Template
# this means 
#LLM Input =
#Instruction
#+ Retrieved Reviews
#+ User Question

In [16]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are an assistant helping users analyze product reviews.

Context:
{context}

Question:
{question}

Provide a concise and accurate answer based only on the context.

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [26]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI assistant that analyzes customer reviews and recommends products.

Using the provided review context, generate the output in the following format.

Here is the summary and recommendations:

Summary:
Provide a short summary of key product feedback (comfort, battery, sound, etc).

Top Recommendations:
List the best 3 products mentioned in the reviews.

Review Sentiment:
Estimate overall sentiment percentage and list common complaints.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [18]:
pip list | findstr langchain

langchain                 1.2.10
langchain-classic         1.0.1
langchain-community       0.4.1
langchain-core            1.2.14
langchain-text-splitters  1.1.1
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# define the llm

In [21]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3",
    temperature=0
)

C:\Users\I335750\AppData\Local\Temp\ipykernel_29176\4194437895.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [19]:
from langchain_classic.chains import RetrievalQA

In [ ]:
# Connect Prompt + LLM + Retriever below
# User Question
     ↓
#Retriever finds relevant reviews
     ↓
#Prompt template inserts them as context
     ↓
#LLM generates answer

#-----------------------------------------------
#You are an assistant helping users analyze product reviews.

#Context:
#This purifier has excellent filtration and low maintenance cost.
#Customers praise its durability and value for money.

#Question:
#Which purifier has the best reviews?

#Answer:

In [27]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

In [28]:
qa_chain.invoke("Which water filter has the best customer reviews?")

{'query': 'Which water filter has the best customer reviews?',
 'result': 'Here is the output in the requested format:\n\nSummary:\nThe majority of customers are satisfied with their experience using Aquacrest and Best Ge MWF refrigerator water filters, praising their effectiveness and great-tasting water. Some users have concerns about the availability and pricing of alternative filter options.\n\nTop Recommendations:\n1. Aquacrest Water Filter\n2. Best Ge MWF Refrigerator Water Filter (Smartwater Compatible Cartridge)\n3. Frigidaire EPTWFU01 Water Filtration Filter\n\nReview Sentiment:\nEstimated overall sentiment: 85% positive, 15% neutral/mixed.\n\nCommon Complaints:\n\n* High price point for alternative filter options\n* Limited availability of aftermarket filters'}

In [29]:
response = qa_chain.invoke("Which water filter has the best customer reviews?")

print(response["result"])

Here is the output in the requested format:

Summary:
The majority of customers are satisfied with their experience using Aquacrest and Best Ge MWF refrigerator water filters, praising their effectiveness and great-tasting water. Some users have concerns about the availability and pricing of alternative filter options.

Top Recommendations:
1. Aquacrest Water Filter
2. Best Ge MWF Refrigerator Water Filter (Smartwater Compatible Cartridge)
3. Frigidaire EPTWFU01 Water Filtration Filter

Review Sentiment:
Estimated overall sentiment: 85% positive, 15% neutral/mixed.

Common Complaints:

* High price point for alternative filter options
* Limited availability of aftermarket filters


In [22]:
##########################################################################################

In [31]:
!pip install rouge-score

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.5 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 6.8 MB/s  0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25025 sha256=990d8a7d5420c03f566870425a21c2a53ddf0c341aec39d74c04e5ce7f6f501f
  Stored in directory: c:\users\i335750\appdata\local\pip\cache\wheels\5f\dd\89\461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score

   ---------------------------------------- 0/3 [absl-py]
   ---------------------------------------- 0/3 [absl-py]
   ----------

In [34]:
# Install rouge-score if not already installed
# !pip install rouge-score

from rouge_score import rouge_scorer

# -------------------------------
# Step 1: Define the query
# -------------------------------
query = "Which water filter has the best customer reviews?"

# -------------------------------
# Step 2: Run the RAG chain
# -------------------------------
response = qa_chain.invoke(query)

# -------------------------------
# Step 3: Extract generated summary from LLM output
# -------------------------------
#generated_summary = response["result"].split("Top Recommendations")[0]
generated_summary = response["result"]

# -------------------------------
# Step 4: Retrieve relevant documents from vector DB
# -------------------------------
docs = retriever.invoke(query)

# -------------------------------
# Step 5: Build reference summary from retrieved context
# -------------------------------
reference_summary = " ".join([doc.page_content for doc in docs])

# Limit size to make comparison fair
reference_summary = reference_summary[:1000]

# -------------------------------
# Step 6: Initialize ROUGE scorer
# -------------------------------
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

# -------------------------------
# Step 7: Calculate ROUGE scores
# -------------------------------
scores = scorer.score(reference_summary, generated_summary)

# -------------------------------
# Step 8: Print results
# -------------------------------
print("Generated Summary:\n")
print(generated_summary)

print("\nReference Summary (from retrieved context):\n")
print(reference_summary)

print("\nROUGE Scores:\n")
print(scores)

Generated Summary:

Here is the output in the requested format:

Summary:
The majority of customers are satisfied with their experience using Aquacrest and Best Ge MWF refrigerator water filters, praising their effectiveness and great-tasting water. Some users have concerns about the availability and pricing of alternative filter options.

Top Recommendations:
1. Aquacrest Water Filter
2. Best Ge MWF Refrigerator Water Filter (Smartwater Compatible Cartridge)
3. Frigidaire EPTWFU01 Water Filtration Filter

Review Sentiment:
Estimated overall sentiment: 85% positive, 15% neutral/mixed.

Common Complaints:

* High price point for alternative filter options
* Limited availability of aftermarket filters

Reference Summary (from retrieved context):

this brand again and can recommend this to anyone looking for a water filter. i received a discount for my honest review. aquacrest   1 4 the price of the smasung filters at home depot or lowes, crazy great tasting water. filter works just as go

The RAG-based summarization system achieved a ROUGE-1 F1 score of 0.37, ROUGE-2 F1 score of 0.15, and ROUGE-L F1 score of 0.20. These results indicate moderate-to-good lexical overlap between the generated summaries and the reference context. 

In [35]:
#------------------------------------

In [37]:
#Semantic similarity

In [36]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

emb1 = model.encode(reference_summary)
emb2 = model.encode(generated_summary)

similarity = cosine_similarity([emb1], [emb2])

print(similarity)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[0.8564595]]


A score of 0.856 indicates:
The generated summary captures the meaning of the reference text well
Even if exact words differ
The model is semantically faithful to the source

sentiment analysis evaluation

In [39]:
df.columns

Index(['parent_asin', 'asin', 'text', 'rating', 'helpful_vote',
       'verified_purchase', 'timestamp', 'review_title', 'product_title',
       'description', 'features', 'categories', 'average_rating',
       'rating_number', 'price', 'store', 'details', 'clean_review_text',
       'clean_product_title', 'clean_features', 'clean_categories',
       'clean_description', 'embedding_text', 'sentiment', 'sentiment_score'],
      dtype='object')

In [41]:
from sklearn.metrics import classification_report

# Create true labels from rating
def rating_to_sentiment(rating):
    if rating >= 4:
        return "POSITIVE"
    elif rating == 3:
        return "NEUTRAL"
    else:
        return "NEGATIVE"

df["true_sentiment"] = df["rating"].apply(rating_to_sentiment)

# Predicted labels from your model
predicted_labels = df["sentiment"]

# True labels from rating
true_labels = df["true_sentiment"]

# Print classification report
print(classification_report(true_labels, predicted_labels))

C:\Users\I335750\AppData\Local\anaconda3\envs\rag_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\I335750\AppData\Local\anaconda3\envs\rag_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

    NEGATIVE       0.29      0.94      0.44      2343
     NEUTRAL       0.00      0.00      0.00      1000
    POSITIVE       0.97      0.72      0.83     16657

    accuracy                           0.71     20000
   macro avg       0.42      0.55      0.42     20000
weighted avg       0.84      0.71      0.74     20000



C:\Users\I335750\AppData\Local\anaconda3\envs\rag_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Lets work towards building UI 

In [43]:
!pip install streamlit

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.1 MB 6.7 MB/s eta 0:00:02
   ------------------- -------------------- 4.5/9.1 MB 15.8 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 18.8 MB/s  0:00:00
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 795.4/795.4 kB 33.3 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------- 6.9/6.9 MB 35.5 MB/s  0:00:00

   ----------------------------------------  0/12 [watchdog]
   ----------------------------------------  0/12 [watchdog]
   ----------------------------------------  0/12 [watchdog]
   --- ------------------------------------  1/12 [toml]
   ------ ---------------------------------  2/12 [smmap]
   ---------- -----------------------------  3/12 [protobuf]
   ---------- -----------------------------  3/12 [p

In [47]:
import matplotlib.pyplot as plt

In [46]:
!pip install matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 6.6 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.1 MB 15.1 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 18.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 40.3 MB/s  0:00:00

   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pyparsing]
   ---------------------------------------- 0/6 [pyparsing]
   ------ --------------------------------- 1/6 [kiwisolver]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- --